# Point to Point Link
---


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import xarray as xr
from hics import HCS, HICSLogger
from hics.plotting import plotnlcd, view_surface_profile, viewcs
from scipy.spatial.transform import Rotation

from xant import XANTLogger, ureg
from xant.antenna import common
from xant.propagation import rflink
from hics.geo.dem import DEM

# Virtual wrap
DEM.virtual_warp = False
# DEM.local_only = True
# XANTLogger.unmute()
# XANTLogger.level = "DEBUG"

# HICSLogger.unmute()
# HICSLogger.level = "DEBUG"
# pv.set_jupyter_backend("trame")

## Setup Coordinate Systems for transmitter and receiver

In [ ]:
llatx = (39.998918, -105.28254, 22)

hcstx = HCS.from_crs((llatx[0] * ureg.degree, llatx[1] * ureg.degree, llatx[2] * ureg.m), hagl=True)

llarx = (39.993457, -105.264, 60)
hcsrx = HCS.from_crs((llarx[0] * ureg.degree, llarx[1] * ureg.degree, llarx[2] * ureg.m), hagl=True)

In [ ]:
mounttx = HCS(
    (0, 0, 0) * ureg.m,
    rotation=Rotation.from_euler("ZXZ", [-111, -91, 0], degrees=True),
    reference=hcstx,
)

mountrx = HCS(
    (0, 0, 0) * ureg.m,
    rotation=Rotation.from_euler("ZXZ", [69, -89, 0], degrees=True),
    reference=hcsrx,
)

In [ ]:
pltter = viewcs(mounttx, hcstx, vector_length=200)
pltter = viewcs(mountrx, hcstx, ax=pltter, vector_length=200)
pltter.show()

In [ ]:
res = view_surface_profile(mounttx, mountrx, aspect=5, time=0)

## Setup RF Parameters and Antennas

In [ ]:
# Frequency and wavelength
f0 = 5.2 * ureg.GHz
lam0 = (ureg.speed_of_light / f0).to("cm")
f0 = [5.2] * ureg.GHz

# Create antenna objects to emulate the Ubiquiti airFiber 5XHD
eta = 0.8
ant0 = common.CircularAperture(lam0 * 6, f0, mounttx) * np.sqrt(eta)
ant1 = common.CircularAperture(lam0 * 6, f0, mountrx) * np.sqrt(eta)

## Compute Propagation Loss with ITM an ITU-R P.1812 Clutter model

In [ ]:
tx_power = 29 * ureg.dBm
res, txcs, rxcs = rflink.calculate_spatial_link(
    ant0,
    tx_power,
    ant1,
    propagation="itm_rflink",
    clutter_method=rflink.propagators.ClutterMethods.ITURP1812,
    clutter_kwargs={"lc_skip_ind": 10},
)
res

## View the Link

In [ ]:
ax = rflink.view_link_horizon(
    ant0,
    ant1,
    res,
    aspect=5,
    k_factor=1.33,
)
ax.grid()
